# Proyecto de Inteligencia Artificial  

## Búsqueda Adversarial en el juego Reversi (Othello)

---

### Profesor  
Hector Jeancarlos Santos Nuesi

---

### Estudiantes  
- Luis Gonzalez 2024-0509
- Robert Yarel Zapata 2024-2010
- Eimy Genao 2024-0543
- Reyli Pineda 2024-0519

1. Configuración inicial

In [ ]:
import time
import math
import random
import copy
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
import pandas as pd
import matplotlib.pyplot as plt

2. Representación del tablero

In [ ]:
EMPTY = 0
BLACK = 1   # Negras
WHITE = -1  # Blancas

DIRECTIONS = [
    (-1, -1), (-1, 0), (-1, 1),
    ( 0, -1),          ( 0, 1),
    ( 1, -1), ( 1, 0), ( 1, 1)
]

BOARD_SIZE = 8

def opponent(player):
    return -player

def create_board():
    board = [[EMPTY for _ in range(BOARD_SIZE)] for _ in range(BOARD_SIZE)]


    board[3][3] = WHITE
    board[3][4] = BLACK
    board[4][3] = BLACK
    board[4][4] = WHITE

    return board

def in_bounds(r, c):
    return 0 <= r < BOARD_SIZE and 0 <= c < BOARD_SIZE

def copy_board(board):
    return [row[:] for row in board]

3. Mostrar el tablero

In [ ]:
def print_board(board):
    print("   " + " ".join(str(i) for i in range(BOARD_SIZE)))
    for i, row in enumerate(board):
        symbols = []
        for cell in row:
            if cell == BLACK:
                symbols.append("B")
            elif cell == WHITE:
                symbols.append("W")
            else:
                symbols.append(".")
        print(f"{i}  " + " ".join(symbols))
    print()

def count_pieces(board):
    black_count = sum(cell == BLACK for row in board for cell in row)
    white_count = sum(cell == WHITE for row in board for cell in row)
    return black_count, white_count

In [ ]:
board = create_board()
print_board(board)
print("Fichas:", count_pieces(board))

   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . W B . . .
4  . . . B W . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .

Fichas: (2, 2)


4. Reglas del juego y movimientos válidos

In [ ]:
def get_flips(board, row, col, player):
    if not in_bounds(row, col) or board[row][col] != EMPTY:
        return []

    all_flips = []

    for dr, dc in DIRECTIONS:
        r, c = row + dr, col + dc
        temp_flips = []

        while in_bounds(r, c) and board[r][c] == opponent(player):
            temp_flips.append((r, c))
            r += dr
            c += dc

        if in_bounds(r, c) and board[r][c] == player and len(temp_flips) > 0:
            all_flips.extend(temp_flips)

    return all_flips

def is_valid_move(board, row, col, player):
    return len(get_flips(board, row, col, player)) > 0

def get_valid_moves(board, player):
    moves = []
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if is_valid_move(board, r, c, player):
                moves.append((r, c))
    return moves

In [ ]:
board = create_board()
print_board(board)
print("Jugadas válidas negras:", get_valid_moves(board, BLACK))
print("Jugadas válidas blancas:", get_valid_moves(board, WHITE))

   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . W B . . .
4  . . . B W . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .

Jugadas válidas negras: [(2, 3), (3, 2), (4, 5), (5, 4)]
Jugadas válidas blancas: [(2, 4), (3, 5), (4, 2), (5, 3)]


5. Aplicar jugadas

In [ ]:
def apply_move(board, move, player):
    row, col = move
    flips = get_flips(board, row, col, player)

    if not flips:
        return None

    new_board = copy_board(board)
    new_board[row][col] = player

    for r, c in flips:
        new_board[r][c] = player

    return new_board

In [ ]:
board = create_board()
print_board(board)

move = (2, 3)
new_board = apply_move(board, move, BLACK)

print("Después de jugar", move)
print_board(new_board)

   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . W B . . .
4  . . . B W . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .

Después de jugar (2, 3)
   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . . . . . .
2  . . . B . . . .
3  . . . B B . . .
4  . . . B W . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .



6. Fin de juego y ganador

In [ ]:
def has_any_moves(board, player):
    return len(get_valid_moves(board, player)) > 0

def game_over(board):
    return not has_any_moves(board, BLACK) and not has_any_moves(board, WHITE)

def get_winner(board):
    black_count, white_count = count_pieces(board)
    if black_count > white_count:
        return BLACK
    elif white_count > black_count:
        return WHITE
    else:
        return 0

7. Heurísticas

Matriz de pesos posicionales

In [ ]:
POSITION_WEIGHTS = [
    [100, -20, 10,  5,  5, 10, -20, 100],
    [-20, -50, -2, -2, -2, -2, -50, -20],
    [10,   -2, -1, -1, -1, -1,  -2,  10],
    [5,    -2, -1, -1, -1, -1,  -2,   5],
    [5,    -2, -1, -1, -1, -1,  -2,   5],
    [10,   -2, -1, -1, -1, -1,  -2,  10],
    [-20, -50, -2, -2, -2, -2, -50, -20],
    [100, -20, 10,  5,  5, 10, -20, 100]
]

Heurística 1: diferencia de fichas

In [ ]:
def heuristic_piece_difference(board, player):
    black_count, white_count = count_pieces(board)
    if player == BLACK:
        return black_count - white_count
    else:
        return white_count - black_count

Heurística 2: movilidad

In [ ]:
def heuristic_mobility(board, player):
    my_moves = len(get_valid_moves(board, player))
    opp_moves = len(get_valid_moves(board, opponent(player)))
    return my_moves - opp_moves

Heurística 3: esquinas

In [ ]:
def heuristic_corners(board, player):
    corners = [(0,0), (0,7), (7,0), (7,7)]
    score = 0
    for r, c in corners:
        if board[r][c] == player:
            score += 1
        elif board[r][c] == opponent(player):
            score -= 1
    return score

Heurística 4: casillas peligrosas cerca de esquinas

In [ ]:
def heuristic_corner_closeness(board, player):
    adjacent_map = {
        (0,0): [(0,1), (1,0), (1,1)],
        (0,7): [(0,6), (1,7), (1,6)],
        (7,0): [(6,0), (7,1), (6,1)],
        (7,7): [(6,7), (7,6), (6,6)]
    }

    score = 0
    for corner, adjacents in adjacent_map.items():
        cr, cc = corner
        if board[cr][cc] == EMPTY:
            for r, c in adjacents:
                if board[r][c] == player:
                    score -= 1
                elif board[r][c] == opponent(player):
                    score += 1
    return score

Heurística 5: pesos posicionales

In [ ]:
def heuristic_position_weights(board, player):
    score = 0
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if board[r][c] == player:
                score += POSITION_WEIGHTS[r][c]
            elif board[r][c] == opponent(player):
                score -= POSITION_WEIGHTS[r][c]
    return score

8. Función de evaluación combinada

In [ ]:
DEFAULT_WEIGHTS_1 = {
    "piece_diff": 1.0,
    "mobility": 4.0,
    "corners": 30.0,
    "corner_closeness": 12.0,
    "position_weights": 2.0
}

DEFAULT_WEIGHTS_2 = {
    "piece_diff": 2.0,
    "mobility": 6.0,
    "corners": 40.0,
    "corner_closeness": 15.0,
    "position_weights": 3.0
}

def evaluate_board(board, player, weights, active_heuristics=5):
    components = [
        ("piece_diff", heuristic_piece_difference(board, player)),
        ("mobility", heuristic_mobility(board, player)),
        ("corners", heuristic_corners(board, player)),
        ("corner_closeness", heuristic_corner_closeness(board, player)),
        ("position_weights", heuristic_position_weights(board, player))
    ]

    score = 0
    for i, (name, value) in enumerate(components[:active_heuristics]):
        score += weights[name] * value

    return score

In [ ]:
board = create_board()
print_board(board)
print("Evaluación para BLACK con 5 heurísticas:", evaluate_board(board, BLACK, DEFAULT_WEIGHTS_1, active_heuristics=5))
print("Evaluación para WHITE con 5 heurísticas:", evaluate_board(board, WHITE, DEFAULT_WEIGHTS_1, active_heuristics=5))

   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . W B . . .
4  . . . B W . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .

Evaluación para BLACK con 5 heurísticas: 0.0
Evaluación para WHITE con 5 heurísticas: 0.0


9. Estadísticas de búsqueda

In [ ]:
@dataclass
class SearchStats:
    nodes_expanded: int = 0
    max_depth_reached: int = 0
    prunings: int = 0

10. Minimax con poda alpha-beta

In [ ]:
def minimax_alpha_beta(board, current_player, root_player, depth, alpha, beta, maximizing, weights, active_heuristics, stats):
    stats.nodes_expanded += 1
    current_level = depth
    stats.max_depth_reached = max(stats.max_depth_reached, current_level)

    if depth == 0 or game_over(board):
        return evaluate_board(board, root_player, weights, active_heuristics), None

    valid_moves = get_valid_moves(board, current_player)

    # Si no hay movimientos válidos, el turno pasa al oponente
    if not valid_moves:
        return minimax_alpha_beta(
            board=board,
            current_player=opponent(current_player),
            root_player=root_player,
            depth=depth - 1,
            alpha=alpha,
            beta=beta,
            maximizing=not maximizing,
            weights=weights,
            active_heuristics=active_heuristics,
            stats=stats
        )

    best_move = None

    if maximizing:
        max_eval = -math.inf

        for move in valid_moves:
            new_board = apply_move(board, move, current_player)

            eval_score, _ = minimax_alpha_beta(
                board=new_board,
                current_player=opponent(current_player),
                root_player=root_player,
                depth=depth - 1,
                alpha=alpha,
                beta=beta,
                maximizing=False,
                weights=weights,
                active_heuristics=active_heuristics,
                stats=stats
            )

            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move

            alpha = max(alpha, max_eval)

            if beta <= alpha:
                stats.prunings += 1
                break

        return max_eval, best_move

    else:
        min_eval = math.inf

        for move in valid_moves:
            new_board = apply_move(board, move, current_player)

            eval_score, _ = minimax_alpha_beta(
                board=new_board,
                current_player=opponent(current_player),
                root_player=root_player,
                depth=depth - 1,
                alpha=alpha,
                beta=beta,
                maximizing=True,
                weights=weights,
                active_heuristics=active_heuristics,
                stats=stats
            )

            if eval_score < min_eval:
                min_eval = eval_score
                best_move = move

            beta = min(beta, min_eval)

            if beta <= alpha:
                stats.prunings += 1
                break

        return min_eval, best_move

11. Búsqueda de profundidad iterativa (IDS)

In [ ]:
def iterative_deepening_search(board, player, max_time, weights, active_heuristics=5, max_depth_limit=20):
    start_time = time.time()

    best_move = None
    best_score = None
    last_completed_depth = 0

    total_stats = SearchStats()

    for depth in range(1, max_depth_limit + 1):
        elapsed = time.time() - start_time
        if elapsed >= max_time:
            break

        current_stats = SearchStats()

        score, move = minimax_alpha_beta(
            board=board,
            current_player=player,
            root_player=player,
            depth=depth,
            alpha=-math.inf,
            beta=math.inf,
            maximizing=True,
            weights=weights,
            active_heuristics=active_heuristics,
            stats=current_stats
        )

        elapsed_after_search = time.time() - start_time
        if elapsed_after_search < max_time:
            best_move = move
            best_score = score
            last_completed_depth = depth

            total_stats.nodes_expanded = current_stats.nodes_expanded
            total_stats.max_depth_reached = current_stats.max_depth_reached
            total_stats.prunings = current_stats.prunings
        else:
            break

    return {
        "move": best_move,
        "score": best_score,
        "depth_reached": last_completed_depth,
        "nodes_expanded": total_stats.nodes_expanded,
        "prunings": total_stats.prunings,
        "time_elapsed": time.time() - start_time
    }

12. Jugadores base

In [ ]:
def random_player(board, player):
    moves = get_valid_moves(board, player)
    if not moves:
        return None
    return random.choice(moves)

In [ ]:
def greedy_player(board, player):
    moves = get_valid_moves(board, player)
    if not moves:
        return None

    best_move = None
    best_value = -math.inf

    for move in moves:
        new_board = apply_move(board, move, player)
        value = heuristic_piece_difference(new_board, player)

        if value > best_value:
            best_value = value
            best_move = move

    return best_move

In [ ]:
def worst_player(board, player):
    moves = get_valid_moves(board, player)
    if not moves:
        return None

    worst_move = None
    worst_value = math.inf

    for move in moves:
        new_board = apply_move(board, move, player)
        value = heuristic_piece_difference(new_board, player)

        if value < worst_value:
            worst_value = value
            worst_move = move

    return worst_move

13. Jugador humano

In [ ]:
def human_player(board, player):
    valid_moves = get_valid_moves(board, player)

    if not valid_moves:
        print("No hay movimientos válidos para este jugador.")
        return None

    print("Movimientos válidos:", valid_moves)

    while True:
        try:
            row = int(input("Ingresa la fila: "))
            col = int(input("Ingresa la columna: "))
            move = (row, col)

            if move in valid_moves:
                return move
            else:
                print("Movimiento inválido. Intenta otra vez.")
        except:
            print("Entrada no válida. Debes escribir números.")

14. Jugador IA

In [ ]:
def ai_player(board, player, max_time=3, weights=DEFAULT_WEIGHTS_1, active_heuristics=5):
    result = iterative_deepening_search(
        board=board,
        player=player,
        max_time=max_time,
        weights=weights,
        active_heuristics=active_heuristics
    )
    return result["move"], result

15. Motor de juego

In [ ]:
def play_game(player_black_func, player_white_func, show_board=True, game_name="Partida"):
    board = create_board()
    current_player = BLACK

    black_search_info = {
        "nodes_expanded": 0,
        "depth_reached": 0,
        "time_elapsed": 0,
        "prunings": 0
    }

    white_search_info = {
        "nodes_expanded": 0,
        "depth_reached": 0,
        "time_elapsed": 0,
        "prunings": 0
    }

    if show_board:
        print(f"=== {game_name} ===")
        print_board(board)

    while not game_over(board):
        valid_moves = get_valid_moves(board, current_player)

        if not valid_moves:
            if show_board:
                print(f"Jugador {'BLACK' if current_player == BLACK else 'WHITE'} sin movimientos. Pasa turno.")
            current_player = opponent(current_player)
            continue

        if current_player == BLACK:
            result = player_black_func(board, BLACK)
        else:
            result = player_white_func(board, WHITE)

        move = None
        info = None

        if isinstance(result, tuple) and len(result) == 2 and isinstance(result[1], dict):
            move, info = result
        else:
            move = result

        if move is not None:
            board = apply_move(board, move, current_player)

            if info:
                if current_player == BLACK:
                    black_search_info["nodes_expanded"] += info["nodes_expanded"]
                    black_search_info["depth_reached"] = max(black_search_info["depth_reached"], info["depth_reached"])
                    black_search_info["time_elapsed"] += info["time_elapsed"]
                    black_search_info["prunings"] += info["prunings"]
                else:
                    white_search_info["nodes_expanded"] += info["nodes_expanded"]
                    white_search_info["depth_reached"] = max(white_search_info["depth_reached"], info["depth_reached"])
                    white_search_info["time_elapsed"] += info["time_elapsed"]
                    white_search_info["prunings"] += info["prunings"]

            if show_board:
                print(f"Jugador {'BLACK' if current_player == BLACK else 'WHITE'} juega: {move}")
                print_board(board)

        current_player = opponent(current_player)

    black_count, white_count = count_pieces(board)
    winner = get_winner(board)

    winner_text = "Empate"
    if winner == BLACK:
        winner_text = "BLACK"
    elif winner == WHITE:
        winner_text = "WHITE"

    if show_board:
        print("Juego terminado")
        print("Fichas BLACK:", black_count)
        print("Fichas WHITE:", white_count)
        print("Ganador:", winner_text)

    return {
        "winner": winner_text,
        "black_score": black_count,
        "white_score": white_count,
        "black_stats": black_search_info,
        "white_stats": white_search_info,
        "final_board": board
    }

16. Configuraciones de IA para benchmark

In [ ]:
def ai_config_1_time_1(board, player):
    return ai_player(board, player, max_time=0.5, weights=DEFAULT_WEIGHTS_1, active_heuristics=3)

def ai_config_1_time_3(board, player):
    return ai_player(board, player, max_time=1, weights=DEFAULT_WEIGHTS_1, active_heuristics=3)

def ai_config_1_time_10(board, player):
    return ai_player(board, player, max_time=2, weights=DEFAULT_WEIGHTS_1, active_heuristics=3)

def ai_config_2_time_3(board, player):
    return ai_player(board, player, max_time=1, weights=DEFAULT_WEIGHTS_2, active_heuristics=3)


def ai_1_heuristic(board, player):
    return ai_player(board, player, max_time=0.5, weights=DEFAULT_WEIGHTS_1, active_heuristics=1)

def ai_2_heuristics(board, player):
    return ai_player(board, player, max_time=0.5, weights=DEFAULT_WEIGHTS_1, active_heuristics=2)

def ai_3_heuristics(board, player):
    return ai_player(board, player, max_time=0.5, weights=DEFAULT_WEIGHTS_1, active_heuristics=3)

def ai_4_heuristics(board, player):
    return ai_player(board, player, max_time=0.5, weights=DEFAULT_WEIGHTS_1, active_heuristics=4)

def ai_5_heuristics(board, player):
    return ai_player(board, player, max_time=0.5, weights=DEFAULT_WEIGHTS_1, active_heuristics=5)

17. Pruebas rápidas

In [ ]:
resultado = play_game(
    player_black_func=ai_config_1_time_3,
    player_white_func=random_player,
    show_board=False,
    game_name="IA vs Random"
)

resultado

KeyboardInterrupt: 

In [ ]:
resultado = play_game(
    player_black_func=ai_config_1_time_3,
    player_white_func=greedy_player,
    show_board=False,
    game_name="IA vs Greedy"
)

resultado

In [ ]:
resultado = play_game(
    player_black_func=ai_config_1_time_3,
    player_white_func=worst_player,
    show_board=False,
    game_name="IA vs Worst"
)

resultado

18. Benchmark automático

In [ ]:
def run_benchmark(num_games=1):
    ai_variants = {
        "Pesos_1": ai_config_1_time_1,
        "Pesos_2": ai_config_2_time_3,
        "1_heuristica": ai_1_heuristic,
        "2_heuristicas": ai_2_heuristics,
        "3_heuristicas": ai_3_heuristics,
        "4_heuristicas": ai_4_heuristics,
        "5_heuristicas": ai_5_heuristics,
        "Tiempo_rapido": ai_config_1_time_1,
        "Tiempo_medio": ai_config_1_time_3
    }

    opponents = {
        "Random": random_player,
        "Greedy": greedy_player
    }

    rows = []

    for ai_name, ai_func in ai_variants.items():
        for opp_name, opp_func in opponents.items():
            for game_idx in range(1, num_games + 1):
                print(f"Ejecutando: {ai_name} vs {opp_name} | partida {game_idx}")

                result = play_game(
                    player_black_func=ai_func,
                    player_white_func=opp_func,
                    show_board=False,
                    game_name=f"{ai_name} vs {opp_name}"
                )

                rows.append({
                    "AI": ai_name,
                    "Opponent": opp_name,
                    "Game": game_idx,
                    "Winner": result["winner"],
                    "BlackScore": result["black_score"],
                    "WhiteScore": result["white_score"],
                    "NodesExpanded": result["black_stats"]["nodes_expanded"],
                    "DepthReached": result["black_stats"]["depth_reached"],
                    "TimeElapsed": result["black_stats"]["time_elapsed"],
                    "Prunings": result["black_stats"]["prunings"]
                })

    return pd.DataFrame(rows)

19. Resumen del benchmark

In [ ]:
df_results = run_benchmark(num_games=1)
df_results.head()

In [ ]:
summary_df = df_results.groupby(["AI", "Opponent"], as_index=False).agg({
    "BlackScore": "mean",
    "WhiteScore": "mean",
    "NodesExpanded": "mean",
    "DepthReached": "mean",
    "TimeElapsed": "mean",
    "Prunings": "mean"
})

summary_df

20. Gráficas de análisis

In [ ]:
if summary_df.empty:
    print("summary_df está vacío. Ejecuta primero la celda del benchmark.")
else:
    plt.figure(figsize=(12, 6))
    for opponent in summary_df["Opponent"].unique():
        subset = summary_df[summary_df["Opponent"] == opponent]
        plt.plot(subset["AI"], subset["BlackScore"], marker="o", label=opponent)

    plt.xticks(rotation=45)
    plt.title("Puntos promedio obtenidos por la IA")
    plt.xlabel("Configuración de IA")
    plt.ylabel("Puntos promedio")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
if summary_df.empty:
    print("summary_df está vacío. Ejecuta primero la celda del benchmark.")
else:
    plt.figure(figsize=(12, 6))
    for opponent in summary_df["Opponent"].unique():
        subset = summary_df[summary_df["Opponent"] == opponent]
        plt.plot(subset["AI"], subset["NodesExpanded"], marker="o", label=opponent)

    plt.xticks(rotation=45)
    plt.title("Nodos expandidos por configuración")
    plt.xlabel("Configuración de IA")
    plt.ylabel("Promedio de nodos expandidos")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
if summary_df.empty:
    print("summary_df está vacío. Ejecuta primero la celda del benchmark.")
else:
    plt.figure(figsize=(12, 6))
    for opponent in summary_df["Opponent"].unique():
        subset = summary_df[summary_df["Opponent"] == opponent]
        plt.plot(subset["AI"], subset["TimeElapsed"], marker="o", label=opponent)

    plt.xticks(rotation=45)
    plt.title("Tiempo de ejecución promedio")
    plt.xlabel("Configuración de IA")
    plt.ylabel("Tiempo promedio (s)")
    plt.legend()
    plt.grid(True)
    plt.show()

21. IA vs IA

In [ ]:
resultado_ai_vs_ai = play_game(
    player_black_func=ai_config_1_time_3,
    player_white_func=ai_config_1_time_3,
    show_board=False,
    game_name="IA vs IA"
)

resultado_ai_vs_ai

22. Humano vs IA

In [ ]:
resultado_humano_vs_ai = play_game(
    player_black_func=human_player,
    player_white_func=ai_config_1_time_3,
    show_board=True,
    game_name="Humano vs IA"
)

=== Humano vs IA ===
   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . W B . . .
4  . . . B W . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .

Movimientos válidos: [(2, 3), (3, 2), (4, 5), (5, 4)]
